# Homework 2 - Machine Learning for Regression

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("car_fuel_efficiency_2026.csv")
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


## Preparing the dataset

In [2]:
base = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
data = df[base + ['fuel_efficiency_mpg']].copy()
data.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


## EDA


In [3]:
data['fuel_efficiency_mpg'].describe()

count    10000.000000
mean        29.997700
std          2.930245
min         19.800000
25%         28.000000
50%         30.000000
75%         31.900000
max         41.200000
Name: fuel_efficiency_mpg, dtype: float64

## Q1. Missing values

In [4]:
data[base].isna().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
dtype: int64

## Q2. Median of horsepower



In [5]:
data['horsepower'].median()

np.float64(254.0)

## Train/validation/test split (as in the lecture: 60/20/20)



In [6]:
def split_data(data, seed):
    n = len(data)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = data.iloc[idx[:n_train]]
    df_val = data.iloc[idx[n_train:n_train + n_val]]
    df_test = data.iloc[idx[n_train + n_val:]]
    return df_train, df_val, df_test

df_train, df_val, df_test = split_data(data, seed=42)
print(len(df_train), len(df_val), len(df_test))

6000 2000 2000


## Helper functions (from the lessons)

In [7]:
def train_linear_regression(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    regularizer = r * np.eye(XTX.shape[0])
    regularizer[0, 0] = 0.0

    XTX_inv = np.linalg.inv(XTX + regularizer)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]


def prepare_X(df, fill):
    X = df[base].fillna(fill).values
    return X


def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)


def score_rmse(df_tr, df_va, fill, r=0.0):
    X_tr = prepare_X(df_tr, fill)
    w0, w = train_linear_regression(X_tr, df_tr['fuel_efficiency_mpg'].values, r)
    X_va = prepare_X(df_va, fill)
    y_pred = w0 + X_va.dot(w)
    return rmse(df_va['fuel_efficiency_mpg'].values, y_pred)

## Q3. Fill missing values: 0 vs mean



In [8]:
rmse_zero = score_rmse(df_train, df_val, fill=0.0)
mean_hp = df_train['horsepower'].mean()
rmse_mean = score_rmse(df_train, df_val, fill=mean_hp)

print('With 0   :', round(rmse_zero, 3))
print('With mean:', round(rmse_mean, 3))
print('-> Better:', 'With mean' if rmse_mean < rmse_zero else 'With 0')

With 0   : 2.205
With mean: 2.202
-> Better: With mean


## Q4. Regularization


In [9]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    s = score_rmse(df_train, df_val, fill=0.0, r=r)
    print(f'r = {r:>5}  RMSE = {round(s, 4)}')

best_r = min([0, 0.01, 0.1, 1, 5, 10, 100], key=lambda r: (round(score_rmse(df_train, df_val, 0.0, r), 4), r))
print('-> Best r:', best_r)

r =     0  RMSE = 2.2053
r =  0.01  RMSE = 2.2053
r =   0.1  RMSE = 2.2053
r =     1  RMSE = 2.2053
r =     5  RMSE = 2.2053
r =    10  RMSE = 2.2053
r =   100  RMSE = 2.2053
-> Best r: 0


## Q5. Influence of the seed



In [10]:
scores = []
for seed in range(10):
    tr, va, _ = split_data(data, seed=seed)
    scores.append(score_rmse(tr, va, fill=0.0))

print('RMSE scores:', [round(s, 3) for s in scores])
std = float(np.std(scores))
print('-> std:', round(std, 3))

RMSE scores: [np.float64(2.239), np.float64(2.202), np.float64(2.163), np.float64(2.204), np.float64(2.202), np.float64(2.246), np.float64(2.27), np.float64(2.191), np.float64(2.217), np.float64(2.207)]
-> std: 0.029


## Q6. Train on train+validation, test on test


In [11]:
tr9, va9, te9 = split_data(data, seed=9)
combo = pd.concat([tr9, va9])

X_combo = prepare_X(combo, 0.0)
w0, w = train_linear_regression(X_combo, combo['fuel_efficiency_mpg'].values, r=0.001)

X_test = prepare_X(te9, 0.0)
y_pred = w0 + X_test.dot(w)
print('-> RMSE on test:', round(rmse(te9['fuel_efficiency_mpg'].values, y_pred), 3))

-> RMSE on test: 2.236
